THERMAL

CLEANING

In [18]:
!unzip Thermal.zip


Archive:  Thermal.zip
   creating: Thermal/
   creating: Thermal/train/
   creating: Thermal/train/Control Group/
  inflating: Thermal/train/Control Group/CG009_M_R-rotated1-rotated1.png  
  inflating: Thermal/train/Control Group/CG009_M_R-rotated1-rotated2.png  
  inflating: Thermal/train/Control Group/CG009_M_R-rotated1.png  
  inflating: Thermal/train/Control Group/CG009_M_R-rotated2-rotated1.png  
  inflating: Thermal/train/Control Group/CG009_M_R-rotated2-rotated2.png  
  inflating: Thermal/train/Control Group/CG009_M_R-rotated2.png  
  inflating: Thermal/train/Control Group/CG009_M_R-sharpened-rotated1.png  
  inflating: Thermal/train/Control Group/CG009_M_R-sharpened-rotated2.png  
  inflating: Thermal/train/Control Group/CG009_M_R-sharpened.png  
  inflating: Thermal/train/Control Group/CG009_M_R.png  
  inflating: Thermal/train/Control Group/CG010_M_L-rotated1-rotated1.png  
  inflating: Thermal/train/Control Group/CG010_M_L-rotated1-rotated2.png  
  inflating: Thermal/train/C

In [19]:

import os
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

THERMAL_PATH = "/content/Thermal"

In [22]:
# Seperate and Count Images
def count_images():

    thermal = Path(THERMAL_PATH)

    for split in ['train', 'val']:
        print(f"\n{split.upper()}:")
        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if folder.exists():
                images = [f for f in folder.iterdir() if f.suffix.lower() in ['.jpg', '.png', '.bmp', '.tif']]
                print(f"  {group}: {len(images)} images")
            else:
                print(f"  {group}: Folder not found!")

count_images()


TRAIN:
  Control Group: 720 images
  DM Group: 724 images

VAL:
  Control Group: 170 images
  DM Group: 252 images


In [23]:
# Check Resolutions
def check_resolutions():

    thermal = Path(THERMAL_PATH)
    resolutions = {}

    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.bmp', '.tif']:
                    continue

                img = cv2.imread(str(img_path))
                if img is not None:
                    h, w = img.shape[:2]
                    res = f"{w}x{h}"
                    resolutions[res] = resolutions.get(res, 0) + 1

    print("\nResolutions found:")
    for res, count in resolutions.items():
        print(f"  {res}: {count} images")

    if len(resolutions) == 1:
        print("\nAll images have consistent resolution!")
    else:
        print("\nMultiple resolutions found - need standardization")

check_resolutions()


Resolutions found:
  224x224: 1533 images
  60x142: 1 images
  76x157: 1 images
  48x119: 1 images
  66x160: 1 images
  55x125: 1 images
  53x144: 1 images
  75x178: 1 images
  54x125: 1 images
  76x183: 1 images
  64x163: 2 images
  72x170: 1 images
  67x177: 1 images
  72x185: 1 images
  68x154: 1 images
  88x200: 1 images
  69x184: 1 images
  70x186: 1 images
  60x121: 2 images
  65x140: 1 images
  59x116: 1 images
  50x149: 1 images
  73x176: 2 images
  42x109: 1 images
  76x165: 1 images
  59x152: 2 images
  56x135: 1 images
  41x106: 1 images
  67x171: 1 images
  65x139: 1 images
  52x132: 1 images
  67x156: 1 images
  58x120: 1 images
  57x141: 1 images
  67x172: 3 images
  60x150: 1 images
  63x119: 1 images
  73x182: 1 images
  76x164: 1 images
  60x160: 1 images
  64x128: 1 images
  46x128: 1 images
  68x174: 2 images
  71x176: 1 images
  70x187: 1 images
  68x148: 1 images
  48x113: 1 images
  59x138: 1 images
  57x137: 2 images
  68x169: 1 images
  68x176: 1 images
  56x14

In [24]:
# Check Formats
def check_formats():

    thermal = Path(THERMAL_PATH)
    formats = {}
    color_modes = {}

    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.bmp', '.tif']:
                    continue

                # Format
                fmt = img_path.suffix.lower()
                formats[fmt] = formats.get(fmt, 0) + 1

                # Color mode
                img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
                if img is not None:
                    if len(img.shape) == 2:
                        mode = "Grayscale"
                    elif len(img.shape) == 3:
                        mode = f"{img.shape[2]} channels"
                    else:
                        mode = "Unknown"
                    color_modes[mode] = color_modes.get(mode, 0) + 1

    print("\nFile formats:")
    for fmt, count in formats.items():
        print(f"  {fmt}: {count} images")

    print("\nColor modes:")
    for mode, count in color_modes.items():
        print(f"  {mode}: {count} images")

check_formats()


File formats:
  .png: 1866 images

Color modes:
  3 channels: 1866 images


In [25]:
# check grayscale
def verify_grayscale():

    thermal = Path(THERMAL_PATH)
    non_grayscale = []

    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.bmp', '.tif']:
                    continue

                img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
                if img is not None and len(img.shape) == 3:
                    non_grayscale.append(img_path.name)

    if len(non_grayscale) == 0:
        print("All images are grayscale!")
    else:
        print(f"Found {len(non_grayscale)} non-grayscale images:")
        for name in non_grayscale[:10]:
            print(f"  - {name}")

verify_grayscale()

Found 1866 non-grayscale images:
  - CG022_M_L-sharpened-rotated2.png
  - CG015_M_R-rotated2.png
  - CG016_M_R.png
  - CG026_M_R-rotated2.png
  - CG033_M_R-rotated2-rotated2.png
  - CG034_M_R-sharpened.png
  - CG035_M_L-rotated2-rotated2.png
  - CG025_M_L-rotated1-rotated2.png
  - CG030_M_L.png
  - CG017_F_L.png


In [26]:
# temperature ranges
def temperature_ranges():

    thermal = Path(THERMAL_PATH)
    all_mins = []
    all_maxs = []
    all_means = []

    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.bmp', '.tif']:
                    continue

                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    all_mins.append(img.min())
                    all_maxs.append(img.max())
                    all_means.append(img.mean())

    all_mins = np.array(all_mins)
    all_maxs = np.array(all_maxs)
    all_means = np.array(all_means)

    print("\nTemperature Statistics (Pixel Values):")
    print(f"  Min values: {all_mins.min()} to {all_mins.max()}")
    print(f"  Max values: {all_maxs.min()} to {all_maxs.max()}")
    print(f"  Mean values: {all_means.min():.2f} to {all_means.max():.2f}")
    print(f"\n  Average across all images:")
    print(f"    Min: {all_mins.mean():.2f} (±{all_mins.std():.2f})")
    print(f"    Max: {all_maxs.mean():.2f} (±{all_maxs.std():.2f})")
    print(f"    Mean: {all_means.mean():.2f} (±{all_means.std():.2f})")

    return all_mins, all_maxs, all_means

temperature_ranges()


Temperature Statistics (Pixel Values):
  Min values: 0 to 0
  Max values: 95 to 255
  Mean values: 31.34 to 133.27

  Average across all images:
    Min: 0.00 (±0.00)
    Max: 199.56 (±28.26)
    Mean: 75.88 (±16.67)


(array([0, 0, 0, ..., 0, 0, 0], dtype=uint8),
 array([187, 207, 167, ..., 171, 209, 204], dtype=uint8),
 array([61.57557398, 90.76432956, 71.11596244, ..., 73.91966217,
        63.52432778, 55.04456314]))

In [27]:
# detect outliers
def detect_outliers():

    thermal = Path(THERMAL_PATH)
    image_stats = []

    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() not in ['.jpg', '.png', '.bmp', '.tif']:
                    continue

                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    image_stats.append({
                        'path': img_path,
                        'min': img.min(),
                        'max': img.max(),
                        'mean': img.mean()
                    })

    # Calculate IQR for outlier detection
    maxs = np.array([s['max'] for s in image_stats])
    Q1 = np.percentile(maxs, 25)
    Q3 = np.percentile(maxs, 75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"\nOutlier detection (using IQR method on max values):")
    print(f"  Q1 (25th percentile): {Q1:.2f}")
    print(f"  Q3 (75th percentile): {Q3:.2f}")
    print(f"  IQR: {IQR:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")

    outliers = [s for s in image_stats if s['max'] < lower_bound or s['max'] > upper_bound]

    print(f"\nFound {len(outliers)} outliers ({len(outliers)/len(image_stats)*100:.2f}%)")

    if len(outliers) > 0:
        print("\nOutlier images:")
        for out in outliers[:10]:
            print(f"  {out['path'].name}: max={out['max']:.2f}")

    return outliers

detect_outliers()


Outlier detection (using IQR method on max values):
  Q1 (25th percentile): 176.00
  Q3 (75th percentile): 213.00
  IQR: 37.00
  Lower bound: 120.50
  Upper bound: 268.50

Found 9 outliers (0.48%)

Outlier images:
  DM065_F_L-rotated2.png: max=95.00
  DM059_F_R-rotated1.png: max=118.00
  DM065_F_L-sharpened.png: max=114.00
  DM058_F_R-rotated2.png: max=114.00
  DM065_F_L.png: max=96.00
  DM059_F_R-rotated2.png: max=118.00
  DM058_F_R-rotated1.png: max=113.00
  DM058_F_R.png: max=115.00
  DM065_F_L-rotated1.png: max=95.00


[{'path': PosixPath('/content/Thermal/train/DM Group/DM065_F_L-rotated2.png'),
  'min': np.uint8(0),
  'max': np.uint8(95),
  'mean': np.float64(47.306959502551024)},
 {'path': PosixPath('/content/Thermal/train/DM Group/DM059_F_R-rotated1.png'),
  'min': np.uint8(0),
  'max': np.uint8(118),
  'mean': np.float64(43.034498565051024)},
 {'path': PosixPath('/content/Thermal/train/DM Group/DM065_F_L-sharpened.png'),
  'min': np.uint8(0),
  'max': np.uint8(114),
  'mean': np.float64(52.82816485969388)},
 {'path': PosixPath('/content/Thermal/train/DM Group/DM058_F_R-rotated2.png'),
  'min': np.uint8(0),
  'max': np.uint8(114),
  'mean': np.float64(31.371950733418366)},
 {'path': PosixPath('/content/Thermal/train/DM Group/DM065_F_L.png'),
  'min': np.uint8(0),
  'max': np.uint8(96),
  'mean': np.float64(47.79942922374429)},
 {'path': PosixPath('/content/Thermal/train/DM Group/DM059_F_R-rotated2.png'),
  'min': np.uint8(0),
  'max': np.uint8(118),
  'mean': np.float64(42.62496014030612)},
 {'pa

In [28]:
# FIX

import os
import cv2
import numpy as np
from pathlib import Path
import shutil
from tqdm import tqdm

OUTPUT_PATH = "/content/Thermal_C"


OUTLIER_PREFIXES = ['DM065_F', 'DM059_F', 'DM058_F']
REMOVE_OUTLIERS = True

def clean_thermal_data():

    thermal = Path(THERMAL_PATH)
    output = Path(OUTPUT_PATH)

    print("Creating output directory structure...")
    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            (output / split / group).mkdir(parents=True, exist_ok=True)

    print(f"Output directory created: {output}\n")

    stats = {
        'total': 0,
        'converted_to_grayscale': 0,
        'resized': 0,
        'skipped_outliers': 0,
        'already_correct': 0
    }

    print("Processing images...\n")

    for split in ['train', 'val']:
        print(f"Processing {split.upper()} set...")

        for group in ['Control Group', 'DM Group']:
            folder = thermal / split / group
            if not folder.exists():
                continue

            images = [f for f in folder.iterdir() if f.suffix.lower() == '.png']

            for img_path in tqdm(images, desc=f"  {group}"):
                stats['total'] += 1

                # Check if outlier
                if REMOVE_OUTLIERS:
                    is_outlier = any(img_path.name.startswith(prefix) for prefix in OUTLIER_PREFIXES)
                    if is_outlier:
                        stats['skipped_outliers'] += 1
                        continue

                # Read image
                img = cv2.imread(str(img_path))
                if img is None:
                    print(f"Could not read: {img_path.name}")
                    continue

                original_shape = img.shape
                was_modified = False

                # Step 1: Convert to grayscale if needed
                if len(img.shape) == 3:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                    stats['converted_to_grayscale'] += 1
                    was_modified = True

                # Step 2: Resize to 224x224 if needed
                if img.shape != (224, 224):
                    img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_LINEAR)
                    stats['resized'] += 1
                    was_modified = True

                if not was_modified:
                    stats['already_correct'] += 1

                # Save cleaned image
                output_path = output / split / group / img_path.name
                cv2.imwrite(str(output_path), img)

    # Print statistics
    print("\n" + "=" * 60)
    print("CLEANING COMPLETE!")
    print("=" * 60)
    print(f"\nStatistics:")
    print(f"  Total images processed: {stats['total']}")
    print(f"  Converted to grayscale: {stats['converted_to_grayscale']}")
    print(f"  Resized to 224x224: {stats['resized']}")
    print(f"  Skipped outliers: {stats['skipped_outliers']}")
    print(f"  Already correct: {stats['already_correct']}")
    print(f"\n  Final dataset size: {stats['total'] - stats['skipped_outliers']} images")
    print(f"\nCleaned data saved to: {output}")


def verify_cleaned_data():
    """Verify that all issues are fixed"""

    print("\n" + "=" * 60)
    print("VERIFYING CLEANED DATA")
    print("=" * 60)

    output = Path(OUTPUT_PATH)

    # Check consistency
    resolutions = {}
    color_modes = {}
    total = 0

    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            folder = output / split / group
            if not folder.exists():
                continue

            for img_path in folder.iterdir():
                if img_path.suffix.lower() != '.png':
                    continue

                total += 1
                img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)

                if img is not None:
                    # Resolution
                    h, w = img.shape[:2] if len(img.shape) >= 2 else img.shape
                    res = f"{w}x{h}"
                    resolutions[res] = resolutions.get(res, 0) + 1

                    # Color mode
                    if len(img.shape) == 2:
                        mode = "Grayscale"
                    elif len(img.shape) == 3:
                        mode = f"{img.shape[2]} channels"
                    else:
                        mode = "Unknown"
                    color_modes[mode] = color_modes.get(mode, 0) + 1

    print(f"\nTotal images in cleaned dataset: {total}")

    print(f"\nResolutions:")
    for res, count in resolutions.items():
        status = "Y" if res == "224x224" else "N"
        print(f"  {status} {res}: {count} images")

    print(f"\nColor modes:")
    for mode, count in color_modes.items():
        status = "Y" if mode == "Grayscale" else "N"
        print(f"  {status} {mode}: {count} images")

    # Final check
    all_correct = (len(resolutions) == 1 and "224x224" in resolutions and
                   len(color_modes) == 1 and "Grayscale" in color_modes)

    if all_correct:
        print("\n" + "=" * 60)
        print("ALL ISSUES FIXED! Dataset is clean and ready!")
        print("=" * 60)
    else:
        print("\nSome issues remain - check output above")

if __name__ == "__main__":
    print("=" * 60)
    print("THERMAL DATA CLEANING - FIX ISSUES")
    print("=" * 60)
    print(f"\nInput: {THERMAL_PATH}")
    print(f"Output: {OUTPUT_PATH}")
    print(f"Remove outliers: {REMOVE_OUTLIERS}")

    response = input("\nThis will create a new cleaned dataset. Continue? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Run cleaning
    clean_thermal_data()

    # Verify results
    verify_cleaned_data()

    print("\nDone! Cleaned dataset at:", OUTPUT_PATH)

THERMAL DATA CLEANING - FIX ISSUES

Input: /content/Thermal
Output: /content/Thermal_C
Remove outliers: True

This will create a new cleaned dataset. Continue? (yes/no): yes
Creating output directory structure...
Output directory created: /content/Thermal_C

Processing images...

Processing TRAIN set...


  DM Group: 100%|██████████| 724/724 [00:01<00:00, 448.80it/s]


Processing VAL set...


  DM Group: 100%|██████████| 252/252 [00:00<00:00, 350.11it/s]



CLEANING COMPLETE!

Statistics:
  Total images processed: 1866
  Converted to grayscale: 1842
  Resized to 224x224: 327
  Skipped outliers: 24
  Already correct: 0

  Final dataset size: 1842 images

Cleaned data saved to: /content/Thermal_C

VERIFYING CLEANED DATA

Total images in cleaned dataset: 1842

Resolutions:
  Y 224x224: 1842 images

Color modes:
  Y Grayscale: 1842 images

ALL ISSUES FIXED! Dataset is clean and ready!

Done! Cleaned dataset at: /content/Thermal_C


PREPROCESSING

In [29]:
import os
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [31]:
"""
THERMAL DATA PREPROCESSING - FULL DATASET
Process all thermal images and save preprocessed results
"""

import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

THERMAL_CLEANED_PATH = "/content/Thermal_C"
THERMAL_PREPROCESSED_PATH = "/content/Thermal_P"

APPLY_AUGMENTATION_TO_TRAIN = True
SAVE_AS_NPY = False


def noise_reduction(img):
    """Bilateral filter for noise reduction"""
    return cv2.bilateralFilter(img, 9, 75, 75)

def temperature_calibration(img):
    """Convert pixel values to temperature (°C)"""
    MIN_TEMP = 20.0
    MAX_TEMP = 40.0
    return (img.astype(np.float32) / 255.0) * (MAX_TEMP - MIN_TEMP) + MIN_TEMP

def roi_extraction(img):
    """Extract foot ROI using thresholding and morphology"""
    # Convert to uint8 if needed
    if img.dtype == np.float32 or img.dtype == np.float64:
        img_uint8 = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    else:
        img_uint8 = img.copy()

    # Thresholding
    _, thresh = cv2.threshold(img_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morphological operations
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    closed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel_close)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel_open)

    # Get largest contour
    contours, _ = cv2.findContours(opened, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) > 0:
        largest_contour = max(contours, key=cv2.contourArea)
        mask = np.zeros_like(opened)
        cv2.drawContours(mask, [largest_contour], -1, 255, -1)
    else:
        mask = opened

    # Apply mask
    if img.dtype == np.float32 or img.dtype == np.float64:
        masked_img = img.copy()
        masked_img[mask == 0] = img.min()
    else:
        masked_img = cv2.bitwise_and(img, img, mask=mask)

    return masked_img

def normalization(img):
    """Min-max normalization to [0, 1]"""
    img_min = img.min()
    img_max = img.max()

    if img_max == img_min:
        return np.zeros_like(img, dtype=np.float32)

    return (img.astype(np.float32) - img_min) / (img_max - img_min)

def augmentation(img):
    """Generate augmented versions"""
    augmented = []

    # Original
    augmented.append(('original', img))

    # Horizontal flip
    augmented.append(('flip', cv2.flip(img, 1)))

    # Rotations
    h, w = img.shape[:2]
    center = (w // 2, h // 2)

    for angle in [-10, 10]:
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        augmented.append((f'rot{angle}', rotated))

    # Scaling
    for scale in [0.95, 1.05]:
        new_h, new_w = int(h * scale), int(w * scale)
        scaled = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

        if scale > 1.0:
            start_h = (new_h - h) // 2
            start_w = (new_w - w) // 2
            scaled = scaled[start_h:start_h+h, start_w:start_w+w]
        else:
            pad_h = (h - new_h) // 2
            pad_w = (w - new_w) // 2
            scaled = cv2.copyMakeBorder(scaled, pad_h, h-new_h-pad_h,
                                       pad_w, w-new_w-pad_w,
                                       cv2.BORDER_REFLECT)

        augmented.append((f'scale{scale}', scaled))

    return augmented

def preprocess_image(img, apply_augment=False):
    """Complete preprocessing pipeline"""
    # Step 1: Noise reduction
    denoised = noise_reduction(img)

    # Step 2: Temperature calibration
    calibrated = temperature_calibration(denoised)

    # Step 3: ROI extraction
    roi = roi_extraction(calibrated)

    # Step 4: Normalization
    normalized = normalization(roi)

    # Step 5: Augmentation
    if apply_augment:
        return augmentation(normalized)
    else:
        return normalized

def process_dataset():
    """Process all thermal images"""

    input_path = Path(THERMAL_CLEANED_PATH)
    output_path = Path(THERMAL_PREPROCESSED_PATH)

    # Create output directory
    print("Creating output directory structure...")
    for split in ['train', 'val']:
        for group in ['Control Group', 'DM Group']:
            (output_path / split / group).mkdir(parents=True, exist_ok=True)

    print(f"Output directory: {output_path}\n")

    # Statistics
    stats = {
        'total_processed': 0,
        'train_original': 0,
        'train_augmented': 0,
        'val_processed': 0
    }

    # Process each split
    for split in ['train', 'val']:
        print(f"\nProcessing {split.upper()} set...")

        apply_augment = (split == 'train' and APPLY_AUGMENTATION_TO_TRAIN)

        for group in ['Control Group', 'DM Group']:
            input_folder = input_path / split / group
            output_folder = output_path / split / group

            if not input_folder.exists():
                print(f"{group} folder not found, skipping...")
                continue

            images = list(input_folder.glob('*.png'))
            print(f"  Processing {group}: {len(images)} images...")

            for img_path in tqdm(images, desc=f"    {group}"):
                # Read image
                img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
                if img is None:
                    print(f"Could not read: {img_path.name}")
                    continue

                # Preprocess
                if apply_augment:
                    # Get augmented versions
                    augmented_imgs = preprocess_image(img, apply_augment=True)

                    for aug_name, preprocessed in augmented_imgs:
                        # Create output filename
                        base_name = img_path.stem
                        if aug_name != 'original':
                            output_name = f"{base_name}_{aug_name}"
                        else:
                            output_name = base_name

                        # Save
                        if SAVE_AS_NPY:
                            output_file = output_folder / f"{output_name}.npy"
                            np.save(output_file, preprocessed)
                        else:
                            output_file = output_folder / f"{output_name}.png"
                            # Convert to uint8 for saving as image
                            img_uint8 = (preprocessed * 255).astype(np.uint8)
                            cv2.imwrite(str(output_file), img_uint8)

                        stats['train_augmented'] += 1

                    stats['train_original'] += 1

                else:
                    # No augmentation (validation)
                    preprocessed = preprocess_image(img, apply_augment=False)

                    # Save
                    if SAVE_AS_NPY:
                        output_file = output_folder / f"{img_path.stem}.npy"
                        np.save(output_file, preprocessed)
                    else:
                        output_file = output_folder / f"{img_path.stem}.png"
                        img_uint8 = (preprocessed * 255).astype(np.uint8)
                        cv2.imwrite(str(output_file), img_uint8)

                    stats['val_processed'] += 1

                stats['total_processed'] += 1

    # Print statistics
    print("\n" + "=" * 60)
    print("PREPROCESSING COMPLETE!")
    print("=" * 60)
    print(f"\nStatistics:")
    print(f"  Total original images processed: {stats['total_processed']}")

    if APPLY_AUGMENTATION_TO_TRAIN:
        print(f"  Training images:")
        print(f"    Original: {stats['train_original']}")
        print(f"    After augmentation: {stats['train_augmented']}")
        print(f"    Augmentation factor: {stats['train_augmented'] / stats['train_original']:.1f}x")

    print(f"  Validation images: {stats['val_processed']} (no augmentation)")

    file_format = '.npy' if SAVE_AS_NPY else '.png'
    print(f"\n  File format: {file_format}")
    print(f"  Output location: {output_path}")

    print("\nPreprocessing complete! Ready for feature extraction.")


def verify_preprocessed():
    """Verify preprocessed data"""

    output_path = Path(THERMAL_PREPROCESSED_PATH)

    print("\n" + "=" * 60)
    print("VERIFYING PREPROCESSED DATA")
    print("=" * 60)

    file_ext = '.npy' if SAVE_AS_NPY else '.png'

    for split in ['train', 'val']:
        print(f"\n{split.upper()}:")
        for group in ['Control Group', 'DM Group']:
            folder = output_path / split / group
            if folder.exists():
                files = list(folder.glob(f'*{file_ext}'))
                print(f"  {group}: {len(files)} files")

                # Check a sample file
                if files:
                    sample = files[0]
                    if SAVE_AS_NPY:
                        data = np.load(sample)
                    else:
                        data = cv2.imread(str(sample), cv2.IMREAD_GRAYSCALE)

                    print(f"    Sample: {sample.name}")
                    print(f"    Shape: {data.shape}")
                    print(f"    Range: [{data.min():.4f}, {data.max():.4f}]")
                    print(f"    Mean: {data.mean():.4f}")


if __name__ == "__main__":
    print("=" * 60)
    print("THERMAL DATA PREPROCESSING - FULL DATASET")
    print("=" * 60)
    print(f"\nConfiguration:")
    print(f"  Input: {THERMAL_CLEANED_PATH}")
    print(f"  Output: {THERMAL_PREPROCESSED_PATH}")
    print(f"  Augment training data: {APPLY_AUGMENTATION_TO_TRAIN}")
    print(f"  Save format: {'.npy' if SAVE_AS_NPY else '.png'}")

    response = input("\nStart preprocessing? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Process dataset
    process_dataset()

    # Verify
    verify_preprocessed()

    print("\nAll done! Preprocessed data ready for feature extraction.")

THERMAL DATA PREPROCESSING - FULL DATASET

Configuration:
  Input: /content/Thermal_C
  Output: /content/Thermal_P
  Augment training data: True
  Save format: .png

Start preprocessing? (yes/no): yes
Creating output directory structure...
Output directory: /content/Thermal_P


Processing TRAIN set...
  Processing Control Group: 720 images...


    Control Group: 100%|██████████| 720/720 [00:11<00:00, 64.94it/s]


  Processing DM Group: 700 images...


    DM Group: 100%|██████████| 700/700 [00:10<00:00, 66.41it/s]



Processing VAL set...
  Processing Control Group: 170 images...


    Control Group: 100%|██████████| 170/170 [00:01<00:00, 129.52it/s]


  Processing DM Group: 252 images...


    DM Group: 100%|██████████| 252/252 [00:01<00:00, 126.96it/s]


PREPROCESSING COMPLETE!

Statistics:
  Total original images processed: 1842
  Training images:
    Original: 1420
    After augmentation: 8520
    Augmentation factor: 6.0x
  Validation images: 422 (no augmentation)

  File format: .png
  Output location: /content/Thermal_P

Preprocessing complete! Ready for feature extraction.

VERIFYING PREPROCESSED DATA

TRAIN:
  Control Group: 4320 files
    Sample: CG026_M_R-rotated1-rotated2_rot10.png
    Shape: (224, 224)
    Range: [0.0000, 254.0000]
    Mean: 102.1152
  DM Group: 4200 files
    Sample: DM092_F_R-rotated2_scale1.05.png
    Shape: (224, 224)
    Range: [0.0000, 255.0000]
    Mean: 123.6728

VAL:
  Control Group: 170 files
    Sample: CG006_F_L-rotated1-rotated2.png
    Shape: (224, 224)
    Range: [0.0000, 255.0000]
    Mean: 90.0781
  DM Group: 252 files
    Sample: DM002_M_R-sharpened.png
    Shape: (224, 224)
    Range: [0.0000, 255.0000]
    Mean: 112.1118

All done! Preprocessed data ready for feature extraction.


FEATURE EXTRACTION

In [32]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops

In [33]:
THERMAL_PREPROCESSED_PATH = "/content/Thermal_P"
OUTPUT_CSV = "/content/thermal_features.csv"
LOAD_FROM_NPY = False

In [34]:
def extract_temperature_features(img):

    mean_temp = np.mean(img)
    max_temp = np.max(img)
    std_temp = np.std(img)
    range_temp = max_temp - np.min(img)

    return mean_temp, max_temp, std_temp, range_temp


def extract_spatial_features(img):

    # 1. Hot spot count
    # Define hot spot as pixels > 90th percentile
    threshold = np.percentile(img, 90)
    hot_spots = (img > threshold).astype(np.uint8)

    # Count connected components (hot spot regions)
    num_labels, labels = cv2.connectedComponents(hot_spots)
    hot_spot_count = num_labels - 1  # Subtract background

    # 2. Temperature gradient (using Sobel)
    # Convert to uint8 for Sobel
    img_uint8 = (img * 255).astype(np.uint8)

    # Sobel gradients
    grad_x = cv2.Sobel(img_uint8, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(img_uint8, cv2.CV_64F, 0, 1, ksize=3)

    # Gradient magnitude
    gradient_magnitude = np.sqrt(grad_x**2 + grad_y**2)
    temp_gradient = np.mean(gradient_magnitude)

    # 3. Left-right asymmetry
    h, w = img.shape
    left_half = img[:, :w//2]
    right_half = img[:, w//2:]

    # Flip right half for comparison
    right_half_flipped = np.fliplr(right_half)

    # Calculate asymmetry as mean absolute difference
    # Make sure both halves have same width
    min_width = min(left_half.shape[1], right_half_flipped.shape[1])
    left_crop = left_half[:, :min_width]
    right_crop = right_half_flipped[:, :min_width]

    lr_asymmetry = np.mean(np.abs(left_crop - right_crop))

    return hot_spot_count, temp_gradient, lr_asymmetry


def extract_texture_features(img):

    # Convert to uint8 (0-255) for GLCM
    img_uint8 = (img * 255).astype(np.uint8)

    # Calculate GLCM
    # distances: [1] - look at adjacent pixels
    # angles: [0, np.pi/4, np.pi/2, 3*np.pi/4] - 4 directions
    # levels: 256 - full grayscale range
    distances = [1]
    angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]

    glcm = graycomatrix(img_uint8, distances=distances, angles=angles,
                        levels=256, symmetric=True, normed=True)

    # Extract properties
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]

    # Calculate entropy manually
    # Entropy = -sum(p * log2(p)) where p is probability
    glcm_normalized = glcm / (glcm.sum() + 1e-10)  # Avoid division by zero
    glcm_normalized = glcm_normalized[glcm_normalized > 0]  # Remove zeros
    entropy = -np.sum(glcm_normalized * np.log2(glcm_normalized))

    return contrast, homogeneity, entropy


def extract_all_features(img):

    # Temperature features (4)
    mean_temp, max_temp, std_temp, range_temp = extract_temperature_features(img)

    # Spatial features (3)
    hot_spot_count, temp_gradient, lr_asymmetry = extract_spatial_features(img)

    # Texture features (3)
    glcm_contrast, glcm_homogeneity, glcm_entropy = extract_texture_features(img)

    features = {
        'mean_temp': mean_temp,
        'max_temp': max_temp,
        'std_temp': std_temp,
        'range_temp': range_temp,
        'hot_spot_count': hot_spot_count,
        'temp_gradient': temp_gradient,
        'lr_asymmetry': lr_asymmetry,
        'glcm_contrast': glcm_contrast,
        'glcm_homogeneity': glcm_homogeneity,
        'glcm_entropy': glcm_entropy
    }

    return features


def process_dataset():
    """
    Process all thermal images and extract features
    Save to CSV with filename as identifier
    """

    preprocessed_path = Path(THERMAL_PREPROCESSED_PATH)

    # List to store all features
    all_features = []

    print("=" * 60)
    print("THERMAL FEATURE EXTRACTION")
    print("=" * 60)
    print(f"\nInput: {preprocessed_path}")
    print(f"Output: {OUTPUT_CSV}")

    file_ext = '.npy' if LOAD_FROM_NPY else '.png'

    # Process each split
    for split in ['train', 'val']:
        print(f"\nProcessing {split.upper()} set...")

        for group in ['Control Group', 'DM Group']:
            folder = preprocessed_path / split / group

            if not folder.exists():
                print(f"{group} folder not found, skipping...")
                continue

            # Get all files
            files = list(folder.glob(f'*{file_ext}'))
            print(f"  Processing {group}: {len(files)} images...")

            # Determine label
            label = 'Control' if group == 'Control Group' else 'DM'

            for file_path in tqdm(files, desc=f"    {group}"):
                try:
                    # Load image
                    if LOAD_FROM_NPY:
                        img = np.load(file_path)
                    else:
                        img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
                        img = img.astype(np.float32) / 255.0  # Normalize to [0, 1]

                    if img is None:
                        print(f"Could not load: {file_path.name}")
                        continue

                    # Extract features
                    features = extract_all_features(img)

                    # Add metadata
                    features['filename'] = file_path.name
                    features['split'] = split
                    features['label'] = label

                    all_features.append(features)

                except Exception as e:
                    print(f"Error processing {file_path.name}: {str(e)}")
                    continue

    # Convert to DataFrame
    df = pd.DataFrame(all_features)

    # Reorder columns: metadata first, then features
    column_order = [
        'filename', 'split', 'label',
        'mean_temp', 'max_temp', 'std_temp', 'range_temp',
        'hot_spot_count', 'temp_gradient', 'lr_asymmetry',
        'glcm_contrast', 'glcm_homogeneity', 'glcm_entropy'
    ]
    df = df[column_order]

    # Save to CSV
    output_path = Path(OUTPUT_CSV)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    # Print statistics
    print("\n" + "=" * 60)
    print("FEATURE EXTRACTION COMPLETE!")
    print("=" * 60)
    print(f"\nStatistics:")
    print(f"  Total images processed: {len(df)}")
    print(f"  Train images: {len(df[df['split'] == 'train'])}")
    print(f"  Val images: {len(df[df['split'] == 'val'])}")
    print(f"  Control samples: {len(df[df['label'] == 'Control'])}")
    print(f"  DM samples: {len(df[df['label'] == 'DM'])}")

    print(f"\n  Features extracted: 10")
    print(f"    Temperature: 4 (mean, max, std, range)")
    print(f"    Spatial: 3 (hot spots, gradient, asymmetry)")
    print(f"    Texture: 3 (GLCM contrast, homogeneity, entropy)")

    print(f"\nCSV saved to: {output_path}")

    # Show sample
    print("\nSample of extracted features:")
    print(df.head())

    return df


def verify_features(csv_path):
    """Verify the extracted features"""

    print("\n" + "=" * 60)
    print("VERIFYING FEATURES")
    print("=" * 60)

    df = pd.read_csv(csv_path)

    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")

    print("\nFeature statistics:")
    feature_cols = [
        'mean_temp', 'max_temp', 'std_temp', 'range_temp',
        'hot_spot_count', 'temp_gradient', 'lr_asymmetry',
        'glcm_contrast', 'glcm_homogeneity', 'glcm_entropy'
    ]

    for col in feature_cols:
        print(f"\n{col}:")
        print(f"  Min: {df[col].min():.4f}")
        print(f"  Max: {df[col].max():.4f}")
        print(f"  Mean: {df[col].mean():.4f}")
        print(f"  Std: {df[col].std():.4f}")

    # Check for missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\nMissing values found:")
        print(missing[missing > 0])
    else:
        print("\nNo missing values")

    # Check for duplicates
    duplicates = df.duplicated(subset=['filename']).sum()
    if duplicates > 0:
        print(f"\n{duplicates} duplicate filenames found")
    else:
        print("\nNo duplicate filenames")


if __name__ == "__main__":
    print("=" * 60)
    print("THERMAL FEATURE EXTRACTION - FULL DATASET")
    print("=" * 60)

    response = input("\nStart feature extraction? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        exit()

    # Extract features
    df = process_dataset()

    # Verify
    verify_features(OUTPUT_CSV)

    print("\nFeature extraction complete!")

THERMAL FEATURE EXTRACTION - FULL DATASET

Start feature extraction? (yes/no): yes
THERMAL FEATURE EXTRACTION

Input: /content/Thermal_P
Output: /content/thermal_features.csv

Processing TRAIN set...
  Processing Control Group: 4320 images...


    Control Group: 100%|██████████| 4320/4320 [01:42<00:00, 42.21it/s]


  Processing DM Group: 4200 images...


    DM Group: 100%|██████████| 4200/4200 [01:39<00:00, 42.02it/s]



Processing VAL set...
  Processing Control Group: 170 images...


    Control Group: 100%|██████████| 170/170 [00:04<00:00, 37.85it/s]


  Processing DM Group: 252 images...


    DM Group: 100%|██████████| 252/252 [00:05<00:00, 44.63it/s]



FEATURE EXTRACTION COMPLETE!

Statistics:
  Total images processed: 8942
  Train images: 8520
  Val images: 422
  Control samples: 4490
  DM samples: 4452

  Features extracted: 10
    Temperature: 4 (mean, max, std, range)
    Spatial: 3 (hot spots, gradient, asymmetry)
    Texture: 3 (GLCM contrast, homogeneity, entropy)

CSV saved to: /content/thermal_features.csv

Sample of extracted features:
                                    filename  split    label  mean_temp  \
0      CG026_M_R-rotated1-rotated2_rot10.png  train  Control   0.400452   
1           CG022_M_L-sharpened-rotated2.png  train  Control   0.245989   
2  CG013_M_R-rotated1-rotated2_scale1.05.png  train  Control   0.428803   
3      CG021_M_R-rotated2-rotated2_rot10.png  train  Control   0.303236   
4    CG010_M_R-sharpened-rotated1_rot-10.png  train  Control   0.454267   

   max_temp  std_temp  range_temp  hot_spot_count  temp_gradient  \
0  0.996078  0.301733    0.996078               8      36.625098   
1  1.000000